# Quickstart

Per ADR-002 (2026-04-25, generalized in pass 25), the **recommended
entry point** is `lub.runtime.build_orchestrated_pack`: build calibrated
agents and hand them to an orchestration framework (ruflo, langgraph,
crewai, autogen). The legacy `lub.pipeline.UncertaintyPipeline` shown
below in §3 is the low-level API and remains supported.

This notebook shows both patterns against the deterministic
`DummyBackend` -- no network, no GPU, no API keys.

## 1. Recommended: build an orchestrated pack

An *orchestrated pack* is a list of calibrated agents shaped for any
framework that satisfies `OrchestratorAgentProtocol`. The factory
below produces one; in production the pack is registered with ruflo
via the JSON-RPC bridge or the MCP plugin loader.

In [ ]:
%pip install -e ..

from lub.agents.core import CalibratedAgent
from lub.agents.policies import RefusalPolicy
from lub.runtime import build_orchestrated_pack, OrchestratedAgentSpec
from lub.uncertainty import SelfConsistency
from lub.wrappers.dummy import DummyBackend

class TrivialAgent(CalibratedAgent):
    prompt_template = "Answer concisely: {q}"
    def parse(self, raw: str) -> str:
        return raw.strip()

backend = DummyBackend()
pack = build_orchestrated_pack([
    OrchestratedAgentSpec(
        name="trivial",
        description="Demo calibrated agent on DummyBackend",
        agent_factory=lambda: TrivialAgent(
            backend=backend,
            uncertainty=SelfConsistency(backend, n_samples=4),
            policy=RefusalPolicy(threshold=0.5),
        ),
        tags=("demo",),
    ),
])

for member in pack:
    print(member.name, '->', member.run({"q": "What is 2 + 2?"}))


## 2. Low-level: UncertaintyPipeline

If you don't need orchestration, the original `UncertaintyPipeline`
still works and is identical to one calibrated worker without the
swarm wrapper.

## 3. Install and import (legacy pipeline)

Install from source in editable mode so that the library you just cloned is the one the notebook imports.

In [ ]:
%pip install -e ..

from lub.pipeline import UncertaintyPipeline

## 4. Build a pipeline

`from_pretrained` is the public factory: it resolves the backend string to a class, wires up the estimator, and returns a ready-to-call pipeline.

In [ ]:
pipe = UncertaintyPipeline.from_pretrained(
    model="dummy-model",
    backend="dummy",
    estimator="self_consistency",
    n_samples=8,
    temperature=0.7,
)

## 5. Answer a question

`pipe.answer` delegates to the estimator and returns an `UncertaintyResult`. Inspect `confidence`, `should_refuse`, and the estimator-specific `raw_scores` diagnostic dict.

In [ ]:
result = pipe.answer("What is the Basel III minimum CET1 ratio?")
print("answer    :", result.answer)
print("confidence:", round(result.confidence, 3))
print("refuse?   :", result.should_refuse)
print("raw_scores:", result.raw_scores)